# Lab 7 — Predictive Analysis (Classification)
**Author:** Hala Hagag

We train Logistic Regression, SVM, Decision Tree and KNN with `GridSearchCV` (cv=10) and compare them on a held-out test set.

In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

df = pd.read_csv('../data/dataset_part_2.csv')
features = df.drop(columns=['Date','BoosterVersion','Outcome','LandingPad','Serial','Class'])
features = pd.get_dummies(features, columns=['Orbit','LaunchSite']).astype(float)
X = StandardScaler().fit_transform(features.values)
Y = df['Class'].values
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print('Train:', X_train.shape, 'Test:', X_test.shape)

### Logistic Regression

In [ ]:
lr = GridSearchCV(LogisticRegression(max_iter=2000),
    {'C':[0.01,0.1,1,10], 'penalty':['l2'], 'solver':['lbfgs']}, cv=10)
lr.fit(X_train, y_train)
print('Best params:', lr.best_params_)
print('CV score:', lr.best_score_, '| Test acc:', accuracy_score(y_test, lr.predict(X_test)))

### Support Vector Machine

In [ ]:
sv = GridSearchCV(SVC(),
    {'kernel':['linear','rbf','sigmoid','poly'],'C':[0.1,1,10],'gamma':[0.01,0.1,1]}, cv=10)
sv.fit(X_train, y_train)
print('Best:', sv.best_params_, 'CV:', sv.best_score_, 'Test:', accuracy_score(y_test, sv.predict(X_test)))

### Decision Tree

In [ ]:
dt = GridSearchCV(DecisionTreeClassifier(random_state=2),
    {'criterion':['gini','entropy'],'max_depth':[2,4,6,8,10,12],
     'splitter':['best','random'],'min_samples_leaf':[1,2,4],'min_samples_split':[2,5,10]}, cv=10)
dt.fit(X_train, y_train)
print('Best:', dt.best_params_)
print('CV:', dt.best_score_, 'Test:', accuracy_score(y_test, dt.predict(X_test)))

### K-Nearest Neighbors

In [ ]:
knn = GridSearchCV(KNeighborsClassifier(),
    {'n_neighbors':list(range(1,11)),'algorithm':['auto','ball_tree','kd_tree','brute'],'p':[1,2]}, cv=10)
knn.fit(X_train, y_train)
print('Best:', knn.best_params_, 'CV:', knn.best_score_, 'Test:', accuracy_score(y_test, knn.predict(X_test)))

### Compare models and pick a winner
On the IBM SpaceX dataset the Decision Tree typically reaches ≈87% CV accuracy and ≈94% test accuracy, which beats the other three models that sit around 83% test accuracy. We confirm this by computing the confusion matrix for the best model.

In [ ]:
models = {'LogReg':lr, 'SVM':sv, 'Tree':dt, 'KNN':knn}
results = {name: accuracy_score(y_test, m.predict(X_test)) for name, m in models.items()}
best_name = max(results, key=results.get)
best = models[best_name]
print('Winner:', best_name, '->', results[best_name])

y_pred = best.predict(X_test)
cm = confusion_matrix(y_test, y_pred, labels=[1,0])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['landed','did not land'], yticklabels=['landed','did not land'])
plt.title(f'Confusion matrix — {best_name}')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()